### Reddit Subreddit Rules Evolution for Health Crises

#### Overview
This module collects historical moderation rules from health-related subreddits to study how rules evolved over time during public health crises. The focus is on identifying when rules were introduced, modified, or removed, and how these changes aligned with major events such as the COVID-19 pandemic (2020–2023) and the mpox outbreak (2022).

#### Data Collected
The data is limited to:
- Subreddit rules and guidelines  
- Pinned moderator announcements  
- Policy clarifications issued by moderators  
- Timestamps of rule updates  

#### Objective
The goal is to construct a temporal map of rule evolution to analyze how subreddit governance adapted in response to health misinformation and shifting public health contexts.


In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import json
# Function to save data periodically
def periodicSave(data, date):
    data = pd.DataFrame(data)
    filename = f"./rules{date}.json"

    # Save as JSON (append-friendly)
    with open(filename, "a", encoding="utf-8") as f:
        json.dump(data.to_dict(orient="records"), f, ensure_ascii=False)
        f.write("\n")  # newline separated JSON objects

    print(f"Periodic save completed. Total posts saved: {len(data)}")



In [41]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Path to your saved HTML file
html_file_path = "calendar2023.html"

# Read the HTML file
with open(html_file_path, "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")

# Collect all snapshot links inside calendar-grid
snapshots = []
for a_tag in soup.select(".calendar-grid a"):
    href = a_tag.get("href")
    if href:
        # Full URL
        snapshot_url = urljoin("https://web.archive.org", href)
        
        
        parts = href.split("/")
        if len(parts) > 2 and parts[1] == "web":
            timestamp = parts[2]  # e.g., 20200404
            # Convert to YYYY-MM-DD
            date = f"{timestamp[:4]}-{timestamp[4:6]}-{timestamp[6:8]}"
            snapshots.append((date, snapshot_url))

# Sort by date
snapshots.sort(key=lambda x: x[0])
data=[]
# Print results
for date, url in snapshots:
    snapshotObject={}
    snapshotObject["SnapshotDate"]=date
    snapshotObject["SnapshotLink"]=url
    data.append(snapshotObject)
data=pd.DataFrame(data)
data.to_json("dateSnapshots2023.json")
